In [42]:
import os
print(os.getcwd())

/home/suhak/synthea


In [43]:
import sys
print(sys.executable)

/home/suhak/synthea/venv/bin/python


In [44]:
import pandas as pd
import numpy as np
import json
import glob
from tqdm import tqdm
import time
import os
import gc

In [45]:
#OPERATING_SYS = 'Win'
OPERATING_SYS = 'Linux'

delim = '\\'

if OPERATING_SYS != 'Win':
    delim = '/'

In [46]:
input_root_folder_path = "/home/suhak/hospital_data"

In [47]:
output_folder_path = '/home/suhak/hospital_data/output'


if not os.path.exists(output_folder_path):
    os.makedirs(output_folder_path)
    print("The new Output directory is created!")

In [48]:
#files = glob.glob(input_root_folder_path+delim+'**'+delim+'*.json',recursive=True)

#print('---Found '+str(len(files))+' Json Files---')


files = glob.glob(os.path.join(input_root_folder_path, "**", "*.json"), recursive=True)

print("---Found " + str(len(files)) + " Json Files---")

---Found 2168 Json Files---


In [49]:
def filter_resource(data, resource_type):
    return list(filter(lambda x: x['resource']['resourceType'] == resource_type.strip(), data['entry']))

## Patient

In [50]:
cols = [
    'hospital_name',
    'hospital_city',
    'id',
    'gender',
    'birthDate',
    'maritalStatus',
    'patient_city',
    'state',
    'postalCode',
    'country',
    'deceased',
    'deceasedDateTime'
]

arr = []
start = time.time()
f_count = 0

for file in tqdm(files):
    try:
        with open(file) as f:
            data = json.load(f)

        ar = []

        patient = filter_resource(data, 'Patient')[0]
        orgs = filter_resource(data, 'Organization')

        hospital_name = os.path.basename(os.path.dirname(file))

        if len(orgs) > 0 and 'address' in orgs[0]['resource'] and len(orgs[0]['resource']['address']) > 0:
            hospital_city = orgs[0]['resource']['address'][0].get('city', np.nan)
        else:
            hospital_city = np.nan

        patient_resource = patient['resource']
        patient_address = patient_resource.get('address', [{}])[0]

        ar.append(hospital_name)
        ar.append(hospital_city)
        ar.append(patient_resource.get('id', np.nan))
        ar.append(patient_resource.get('gender', np.nan))
        ar.append(patient_resource.get('birthDate', np.nan))
        ar.append(patient_resource.get('maritalStatus', {}).get('text', np.nan))
        ar.append(patient_address.get('city', np.nan))
        ar.append(patient_address.get('state', np.nan))
        ar.append(patient_address.get('postalCode', np.nan))
        ar.append(patient_address.get('country', np.nan))

        if 'deceasedDateTime' in patient_resource:
            ar.append(True)
            ar.append(patient_resource['deceasedDateTime'])
        else:
            ar.append(False)
            ar.append(np.nan)

        arr.append(ar)

    except Exception as e:
        f_count += 1
        continue

end = time.time()

print(str(f_count) + ' Files Failed...')
print(str(len(arr)) + ' Patient bundles extracted as DataFrame in ' + str(end-start) + ' Seconds')

df_patient = pd.DataFrame(arr, columns=cols)

  0%|          | 0/2168 [00:00<?, ?it/s]

100%|██████████| 2168/2168 [04:11<00:00,  8.61it/s]

8 Files Failed...
2160 Patient bundles extracted as DataFrame in 251.7346088886261 Seconds


In [51]:
print(df_patient['hospital_name'].value_counts())
print(df_patient[['hospital_name', 'hospital_city']].drop_duplicates())
print(df_patient['patient_city'].value_counts().head(20))

hospital_name
MGH              1760
BCMH              290
PortageHealth      89
Aspirus            21
Name: count, dtype: int64
      hospital_name  hospital_city
0               MGH            NaN
1760        Aspirus            NaN
1781           BCMH            NaN
2071  PortageHealth            NaN
patient_city
Marquette    1760
Baraga        290
Houghton       89
Calumet        21
Name: count, dtype: int64


In [52]:
df_patient['deceased'].value_counts()/len(df_patient)

deceased
False    0.843519
True     0.156481
Name: count, dtype: float64

In [53]:
#Dropping Duplicates If Any
df_patient = df_patient.drop_duplicates('id', 
                                        inplace=False, 
                                        ignore_index=True)

In [54]:
df_patient.head()

,hospital_name,hospital_city,id,gender,birthDate,maritalStatus,patient_city,state,postalCode,country,deceased,deceasedDateTime
0,MGH,NaN,209c8400-573b-a7a7-7791-657ca87065ba,female,1951-04-04,Never Married,Marquette,MI,49855,US,False,NaN
1,MGH,NaN,dd494021-27cc-8260-1ac2-fe6cf8324538,female,1970-01-12,Married,Marquette,MI,49855,US,False,NaN
2,MGH,NaN,865ac475-f245-d9a2-02b8-f64676b0d768,female,1983-02-01,Never Married,Marquette,MI,49855,US,False,NaN
3,MGH,NaN,0b4a3b96-af62-3519-7220-684a4c7bf367,male,1953-07-23,Never Married,Marquette,MI,49855,US,False,NaN
4,MGH,NaN,6ef52a27-c55c-4830-220e-ab4fa64ce0b7,female,1997-02-15,Married,Marquette,MI,49855,US,False,NaN


In [55]:
df_patient.to_csv(output_folder_path+delim+'Patient.csv')
del df_patient
gc.collect()

0

## Conditions

In [56]:
cols = ['code','codeText','patientId','encounterId','onsetDateTime','recordedDate','clinicalStatusCode']

arr = []
start = time.time()
f_count = 0

for file in tqdm(files):
    try:
        #load File
        f = open(file)
        data = json.load(f)
        f.close()
        

        conditions = filter_resource(data, 'Condition')
        for cond in conditions:
            ar = []
            
            ar.append(cond['resource']['code']['coding'][0]['code'])
            ar.append(cond['resource']['code']['text'])
            ar.append(cond['resource']['subject']['reference'].strip().split('urn:uuid:')[1])
            ar.append(cond['resource']['encounter']['reference'].strip().split('urn:uuid:')[1])
            ar.append(cond['resource']['onsetDateTime'])
            ar.append(cond['resource']['recordedDate'])
            ar.append(cond['resource']['clinicalStatus']['coding'][0]['code'])
            
            arr.append(ar)
    except Exception as e:
#         print(e)
        f_count += 1
        continue

end = time.time()
print(str(f_count)+' Files Failed...')
print(str(len(arr))+' Patient condition bundles extracted as DataFrame in '+str(end-start)+ 'Seconds')

df_condition = pd.DataFrame(arr, columns = cols)

  0%|          | 0/2168 [00:00<?, ?it/s]

100%|██████████| 2168/2168 [04:01<00:00,  8.99it/s]


0 Files Failed...
91321 Patient condition bundles extracted as DataFrame in 241.23697471618652Seconds


In [57]:
print(df_condition.head())
print(df_condition.shape)
print(df_condition['patientId'].nunique())
print(df_condition['codeText'].value_counts().head(20))

        code                                           codeText  \
0  224295006   Only received primary school education (finding)   
1  714628002                              Prediabetes (finding)   
2  161744009  Past pregnancy history of miscarriage (situation)   
3   87433001                     Pulmonary emphysema (disorder)   
4  162864005            Body mass index 30+ - obesity (finding)   

                              patientId                           encounterId  \
0  209c8400-573b-a7a7-7791-657ca87065ba  209c8400-573b-a7a7-23e4-13686147ca30   
1  209c8400-573b-a7a7-7791-657ca87065ba  209c8400-573b-a7a7-7add-d25aa886e071   
2  209c8400-573b-a7a7-7791-657ca87065ba  209c8400-573b-a7a7-87e7-dac59b014e9c   
3  209c8400-573b-a7a7-7791-657ca87065ba  209c8400-573b-a7a7-7c47-434bef6f2279   
4  209c8400-573b-a7a7-7791-657ca87065ba  209c8400-573b-a7a7-7c47-434bef6f2279   

               onsetDateTime               recordedDate clinicalStatusCode  
0  1969-05-29T01:38:22-04:00  196

In [58]:
df_condition.head()

,code,codeText,patientId,encounterId,onsetDateTime,recordedDate,clinicalStatusCode
0,224295006,Only received primary school education (finding),209c8400-573b-a7a7-7791-657ca87065ba,209c8400-573b-a7a7-23e4-13686147ca30,1969-05-29T01:38:22-04:00,1969-05-29T01:38:22-04:00,active
1,714628002,Prediabetes (finding),209c8400-573b-a7a7-7791-657ca87065ba,209c8400-573b-a7a7-7add-d25aa886e071,1976-06-10T00:50:24-04:00,1976-06-10T00:50:24-04:00,active
2,161744009,Past pregnancy history of miscarriage (situation),209c8400-573b-a7a7-7791-657ca87065ba,209c8400-573b-a7a7-87e7-dac59b014e9c,1986-01-02T16:50:24-05:00,1986-01-02T16:50:24-05:00,active
3,87433001,Pulmonary emphysema (disorder),209c8400-573b-a7a7-7791-657ca87065ba,209c8400-573b-a7a7-7c47-434bef6f2279,1988-04-14T00:50:24-04:00,1988-04-14T00:50:24-04:00,active
4,162864005,Body mass index 30+ - obesity (finding),209c8400-573b-a7a7-7791-657ca87065ba,209c8400-573b-a7a7-7c47-434bef6f2279,1988-04-14T00:50:24-04:00,1988-04-14T00:50:24-04:00,active


In [59]:
df_condition['onsetDateTime'] = pd.to_datetime(df_condition['onsetDateTime'], format="%Y-%m-%dT%H:%M:%S%z", utc=True)
df_condition['recordedDate'] = pd.to_datetime(df_condition['recordedDate'], format="%Y-%m-%dT%H:%M:%S%z", utc=True)

In [60]:
#Extracting resolvedDateTime form Conditions DataFrame

cols = ['patientId','code','encounterId','onsetDateTime','resolvedDateTime','codeText']
arr = []
for name,group in tqdm(df_condition.groupby(['patientId','encounterId','onsetDateTime'])):
    #Groupby Condition Code Again
    for name2, group2 in group.groupby(['code','codeText']):
        ar = []
        # Add patientId
        ar.append(name[0])
        
        # Add code
        ar.append(name2[0])
        
        # Add encounterId
        ar.append(name[1])
        
        # Add onsetDateTime
        ar.append(name[2])
        
        #Get Records with clinicalStatusCode as Resolved
        resolved = group2.query('clinicalStatusCode == "resolved"')
        
        #Add Resolved Date to Array if Resolved Record exists
        if len(resolved) > 0 :
            ar.append(resolved['recordedDate'].max())
        else:
            ar.append(group2['recordedDate'].max())
        
        # Add codeText
        ar.append(name2[1])
        
        arr.append(ar)

df_condition_new = pd.DataFrame(arr, columns = cols)

100%|██████████| 76017/76017 [08:44<00:00, 144.84it/s]


In [61]:
df_condition_new['onsetDateTime'] = pd.to_datetime(df_condition_new['onsetDateTime'], 
                                                   format="%Y-%m-%dT%H:%M:%S%z", utc=True)
df_condition_new['resolvedDateTime'] = pd.to_datetime(df_condition_new['resolvedDateTime'], 
                                                      format="%Y-%m-%dT%H:%M:%S%z", utc=True)
df_condition_new.head()

,patientId,code,encounterId,onsetDateTime,resolvedDateTime,codeText
0,002c0bea-51f6-6d8d-a2b6-49f0fad7189d,10509002,002c0bea-51f6-6d8d-0032-0a25671a4a5e,2017-05-13 22:50:16+00:00,2017-05-13 22:50:16+00:00,Acute bronchitis (disorder)
1,002c0bea-51f6-6d8d-a2b6-49f0fad7189d,314529007,002c0bea-51f6-6d8d-1eee-4737276b56f5,2023-08-08 16:50:16+00:00,2023-08-08 16:50:16+00:00,Medication review due (situation)
2,002c0bea-51f6-6d8d-a2b6-49f0fad7189d,160903007,002c0bea-51f6-6d8d-1eee-4737276b56f5,2023-08-08 17:31:16+00:00,2023-08-08 17:31:16+00:00,Full-time employment (finding)
3,002c0bea-51f6-6d8d-a2b6-49f0fad7189d,224295006,002c0bea-51f6-6d8d-1eee-4737276b56f5,2023-08-08 17:31:16+00:00,2023-08-08 17:31:16+00:00,Only received primary school education (finding)
4,002c0bea-51f6-6d8d-a2b6-49f0fad7189d,73595000,002c0bea-51f6-6d8d-1eee-4737276b56f5,2023-08-08 17:31:16+00:00,2023-08-08 17:31:16+00:00,Stress (finding)


In [62]:
df_condition_new.query('code == "840539006"')

,patientId,code,encounterId,onsetDateTime,resolvedDateTime,codeText
613,0169f4b4-7aff-c59e-4239-f8c2b74083ea,840539006,0169f4b4-7aff-c59e-33a1-85dfff8840e6,2021-09-05 15:23:36+00:00,2021-09-05 15:23:36+00:00,Disease caused by severe acute respiratory syn...
1395,047d7725-95e7-4a47-490b-2b29dac63c30,840539006,047d7725-95e7-4a47-b129-4f2c0fa291ca,2021-04-18 07:45:23+00:00,2021-04-18 07:45:23+00:00,Disease caused by severe acute respiratory syn...
1562,04c47d74-efa3-d42b-f31b-81706484a234,840539006,04c47d74-efa3-d42b-6ac6-605f9a135227,2021-01-18 15:03:02+00:00,2021-01-18 15:03:02+00:00,Disease caused by severe acute respiratory syn...
1616,04dc3274-a4c5-25de-fdea-9fd447cfd20a,840539006,04dc3274-a4c5-25de-ce72-3a6b7ded9f83,2020-11-21 13:39:00+00:00,2020-11-21 13:39:00+00:00,Disease caused by severe acute respiratory syn...
1865,054f6d24-964b-48ea-ad15-a7a2df566550,840539006,054f6d24-964b-48ea-5f0d-41eeff1849e4,2020-10-17 00:23:43+00:00,2020-10-17 00:23:43+00:00,Disease caused by severe acute respiratory syn...
...,...,...,...,...,...,...
87420,f529d549-e9f1-859d-572e-018067b85435,840539006,f529d549-e9f1-859d-cab6-4413fc24e650,2021-02-05 14:14:23+00:00,2021-02-05 14:14:23+00:00,Disease caused by severe acute respiratory syn...
89295,fa555c5f-0e8e-65e6-7999-f04d2988a606,840539006,fa555c5f-0e8e-65e6-eaf6-044e120c061b,2021-09-05 15:07:30+00:00,2021-09-05 15:07:30+00:00,Disease caused by severe acute respiratory syn...
89569,fa8a3f25-8f92-5fb8-a4a5-6bd7c36158b9,840539006,fa8a3f25-8f92-5fb8-ccbf-4d216bc62a6f,2020-12-28 06:18:23+00:00,2020-12-28 06:18:23+00:00,Disease caused by severe acute respiratory syn...
90298,fbd69f67-0d0e-79c2-61e0-e579f5077049,840539006,fbd69f67-0d0e-79c2-b074-92e1a9aac20c,2020-08-24 17:48:25+00:00,2020-08-24 17:48:25+00:00,Disease caused by severe acute respiratory syn...


840539006 Is the Code for COVID 19

In [63]:
df_condition_new.to_csv(output_folder_path+delim+'Condition.csv')
del df_condition_new
gc.collect()

12

## Encounters

In [64]:
cols = ['id','status','code','codeText','start','end','patientId','location','serviceProvider','encounterClass']

arr = []
start = time.time()
f_count = 0

for file in tqdm(files):
    try:
        #load File
        f = open(file)
        data = json.load(f)
        f.close()
        

        encounters = filter_resource(data, 'Encounter')
        for encounter in encounters:
            ar = []
            
            ar.append(encounter['resource']['id'])
            ar.append(encounter['resource']['status'])
            ar.append(encounter['resource']['type'][0]['coding'][0]['code'])
            ar.append(encounter['resource']['type'][0]['text'])
            ar.append(encounter['resource']['period']['start'])
            ar.append(encounter['resource']['period']['end'])
            ar.append(encounter['resource']['subject']['reference'].strip().split('urn:uuid:')[1])
            ar.append(encounter['resource']['location'][0]['location']['display'])
            ar.append(encounter['resource']['serviceProvider']['display'])
            ar.append(encounter['resource']['class']['code'])

            arr.append(ar)
    except Exception as e:
#         print(e)
        f_count += 1
        continue

end = time.time()
print(str(f_count)+' Files Failed...')
print(str(len(arr))+' Patient encounter bundles extracted as DataFrame in '+str(end-start)+ 'Seconds')

df_encounter = pd.DataFrame(arr, columns = cols)

100%|██████████| 2168/2168 [04:14<00:00,  8.53it/s]


0 Files Failed...
147385 Patient encounter bundles extracted as DataFrame in 254.22377920150757Seconds


In [65]:
df_encounter.head()

,id,status,code,codeText,start,end,patientId,location,serviceProvider,encounterClass
0,209c8400-573b-a7a7-de7c-35b4dd6c2010,finished,185347001,Encounter for problem (procedure),1967-12-31T23:50:24-05:00,1968-01-01T01:10:05-05:00,209c8400-573b-a7a7-7791-657ca87065ba,DLP MARQUETTE GENERAL HOSPITAL LLC,DLP MARQUETTE GENERAL HOSPITAL LLC,AMB
1,209c8400-573b-a7a7-23e4-13686147ca30,finished,162673000,General examination of patient (procedure),1969-05-29T00:50:24-04:00,1969-05-29T01:38:22-04:00,209c8400-573b-a7a7-7791-657ca87065ba,BAKER COMMUNITY HEALTH CENTER INC,BAKER COMMUNITY HEALTH CENTER INC,AMB
2,209c8400-573b-a7a7-7add-d25aa886e071,finished,162673000,General examination of patient (procedure),1976-06-10T00:50:24-04:00,1976-06-10T01:25:43-04:00,209c8400-573b-a7a7-7791-657ca87065ba,BAKER COMMUNITY HEALTH CENTER INC,BAKER COMMUNITY HEALTH CENTER INC,AMB
3,209c8400-573b-a7a7-87e7-dac59b014e9c,finished,424619006,Prenatal visit (regime/therapy),1986-01-02T16:50:24-05:00,1986-01-02T17:05:24-05:00,209c8400-573b-a7a7-7791-657ca87065ba,DLP MARQUETTE GENERAL HOSPITAL LLC,DLP MARQUETTE GENERAL HOSPITAL LLC,AMB
4,209c8400-573b-a7a7-7c47-434bef6f2279,finished,702927004,Urgent care clinic (environment),1988-04-14T00:50:24-04:00,1988-04-14T01:33:19-04:00,209c8400-573b-a7a7-7791-657ca87065ba,BAYSIDE DOCS URGENT CARE PLC,BAYSIDE DOCS URGENT CARE PLC,AMB


In [66]:
df_encounter['encounterClass'].value_counts()

encounterClass
AMB     135964
EMER      6556
IMP       3050
HH        1345
VR         470
Name: count, dtype: int64

In [67]:
df_encounter.to_csv(output_folder_path+delim+'Encounter.csv')
del df_encounter
gc.collect()

0

## Observations

In [68]:
cols = ['id','patientId','issuedDate','effectiveDateTime','category','encounter','code','codeText','value','units','snomedCode','observationType']

arr = []
start = time.time()
f_count = 0

for file in tqdm(files):
    try:
        #load File
        f = open(file)
        data = json.load(f)
        f.close()
        

        observations = filter_resource(data, 'Observation')
        for observation in observations:
            
            if 'component' in observation['resource'].keys():
                for comp in observation['resource']['component']:
                    ar = []
                    ar.append(observation['resource']['id'])
                    ar.append(observation['resource']['subject']['reference'].strip().split('urn:uuid:')[1])
                    ar.append(observation['resource']['issued'])
                    ar.append(observation['resource']['effectiveDateTime'])
                    ar.append(observation['resource']['category'][0]['coding'][0]['code'])
                    ar.append(observation['resource']['encounter']['reference'].strip().split('urn:uuid:')[1])
                    
                    ar.append(comp['code']['coding'][0]['code'])
                    ar.append(comp['code']['coding'][0]['display'])

                    if 'valueCodeableConcept' in comp.keys():
                        ar.append(comp['valueCodeableConcept']['coding'][0]['display'])
                        ar.append(np.nan)
                        ar.append(comp['valueCodeableConcept']['coding'][0]['code'])
                        ar.append('text')
                    elif 'valueQuantity' in comp.keys():
                        ar.append(comp['valueQuantity']['value'])
                        ar.append(comp['valueQuantity']['unit'])
                        ar.append(np.nan)
                        ar.append('numeric')
                    else:
                        ar.append(comp['valueString'])
                        ar.append(np.nan)
                        ar.append(np.nan)
                        ar.append('text')

                    arr.append(ar)
            else:
                ar = []
                ar.append(observation['resource']['id'])
                ar.append(observation['resource']['subject']['reference'].strip().split('urn:uuid:')[1])
                ar.append(observation['resource']['issued'])
                ar.append(observation['resource']['effectiveDateTime'])
                ar.append(observation['resource']['category'][0]['coding'][0]['code'])
                ar.append(observation['resource']['encounter']['reference'].strip().split('urn:uuid:')[1])
                
                ar.append(observation['resource']['code']['coding'][0]['code'])
                ar.append(observation['resource']['code']['coding'][0]['display'])

                if 'valueCodeableConcept' in observation['resource'].keys():
                    ar.append(observation['resource']['valueCodeableConcept']['coding'][0]['display'])
                    ar.append(np.nan)
                    ar.append(observation['resource']['valueCodeableConcept']['coding'][0]['code'])
                    ar.append('text')
                elif 'valueString' in observation['resource'].keys():
                    ar.append(observation['resource']['valueString'])
                    ar.append(np.nan)
                    ar.append(np.nan)
                    ar.append('text')
                else:
                    ar.append(observation['resource']['valueQuantity']['value'])
                    ar.append(observation['resource']['valueQuantity']['unit'])
                    ar.append(np.nan)
                    ar.append('numeric')

                arr.append(ar)
    except Exception as e:
#         print(e)
#         print(observation['resource'])
        f_count += 1
        continue

end = time.time()
print(str(f_count)+' Files Failed...')
print(str(len(arr))+' Patient observation bundles extracted as DataFrame in '+str(end-start)+ 'Seconds')

df_observation = pd.DataFrame(arr, columns = cols)

100%|██████████| 2168/2168 [05:55<00:00,  6.10it/s]


6 Files Failed...
2011746 Patient observation bundles extracted as DataFrame in 355.40418553352356Seconds


In [69]:
df_observation

,id,patientId,issuedDate,effectiveDateTime,category,encounter,code,codeText,value,units,snomedCode,observationType
0,209c8400-573b-a7a7-976f-3c385ab35e86,209c8400-573b-a7a7-7791-657ca87065ba,2016-11-30T23:50:24.817-05:00,2016-11-30T23:50:24-05:00,procedure,209c8400-573b-a7a7-e166-7a2fbdad00ac,19926-5,FEV1/FVC,10.968,%,NaN,numeric
1,209c8400-573b-a7a7-8516-3b7781b31876,209c8400-573b-a7a7-7791-657ca87065ba,2016-11-30T23:50:24.817-05:00,2016-11-30T23:50:24-05:00,laboratory,209c8400-573b-a7a7-e166-7a2fbdad00ac,4548-4,Hemoglobin A1c/Hemoglobin.total in Blood,5.96,%,NaN,numeric
2,209c8400-573b-a7a7-f009-5bd8cf69f590,209c8400-573b-a7a7-7791-657ca87065ba,2016-11-30T23:50:24.817-05:00,2016-11-30T23:50:24-05:00,vital-signs,209c8400-573b-a7a7-e166-7a2fbdad00ac,8302-2,Body Height,163.3,cm,NaN,numeric
3,209c8400-573b-a7a7-c491-307b29a8aace,209c8400-573b-a7a7-7791-657ca87065ba,2016-11-30T23:50:24.817-05:00,2016-11-30T23:50:24-05:00,vital-signs,209c8400-573b-a7a7-e166-7a2fbdad00ac,72514-3,Pain severity - 0-10 verbal numeric rating [Sc...,4,{score},NaN,numeric
4,209c8400-573b-a7a7-8320-e800910c7a69,209c8400-573b-a7a7-7791-657ca87065ba,2016-11-30T23:50:24.817-05:00,2016-11-30T23:50:24-05:00,vital-signs,209c8400-573b-a7a7-e166-7a2fbdad00ac,29463-7,Body Weight,74.2,kg,NaN,numeric
...,...,...,...,...,...,...,...,...,...,...,...,...
2011741,94e93cc8-fb91-b3fa-5cfb-50b2e9eef8fc,94e93cc8-fb91-b3fa-f102-bd49abb14d72,2025-04-19T00:19:19.823-04:00,2025-04-19T00:19:19-04:00,survey,94e93cc8-fb91-b3fa-12a0-703d55b97882,32624-9,Race,White,NaN,LA4457-3,text
2011742,94e93cc8-fb91-b3fa-5cfb-50b2e9eef8fc,94e93cc8-fb91-b3fa-f102-bd49abb14d72,2025-04-19T00:19:19.823-04:00,2025-04-19T00:19:19-04:00,survey,94e93cc8-fb91-b3fa-12a0-703d55b97882,56051-6,Hispanic or Latino,No,NaN,LA32-8,text
2011743,94e93cc8-fb91-b3fa-0df0-c0eebd0103e6,94e93cc8-fb91-b3fa-f102-bd49abb14d72,2025-04-19T00:48:23.823-04:00,2025-04-19T00:48:23-04:00,survey,94e93cc8-fb91-b3fa-12a0-703d55b97882,76504-0,Total score [HARK],0,{score},NaN,numeric
2011744,94e93cc8-fb91-b3fa-1cf7-8829fd549004,94e93cc8-fb91-b3fa-f102-bd49abb14d72,2025-04-19T01:25:23.823-04:00,2025-04-19T01:25:23-04:00,survey,94e93cc8-fb91-b3fa-12a0-703d55b97882,55758-7,Patient Health Questionnaire 2 item (PHQ-2) to...,2,{score},NaN,numeric


In [70]:
df_observation.to_csv(output_folder_path+delim+'Observation.csv')
del df_observation
del ar
del arr
gc.collect()

0

## Care Plan

In [71]:
cols = ['id','status','patientId','start','end','category','code','codeText','intent','encounter','careTeam','activityCode','activityCodeText','activityStatus','activityLocation']

arr = []
start = time.time()
f_count = 0

for file in tqdm(files):
    try:
        #load File
        f = open(file)
        data = json.load(f)
        f.close()
        

        cps = filter_resource(data, 'CarePlan')
        for cp in cps:
            if 'activity' in cp['resource'].keys():
                for activity in cp['resource']['activity']:
                    ar = []
                    ar.append(cp['resource']['id'])
                    ar.append(cp['resource']['status'])
                    ar.append(cp['resource']['subject']['reference'].strip().split('urn:uuid:')[1])
                    ar.append(cp['resource']['period']['start'])

                    if 'end' in cp['resource']['period'].keys():
                        ar.append(cp['resource']['period']['end'])
                    else:
                        ar.append(np.nan)

                    ar.append(cp['resource']['category'][0]['coding'][0]['code'])
                    ar.append(cp['resource']['category'][1]['coding'][0]['code'])
                    ar.append(cp['resource']['category'][1]['coding'][0]['display'])
                    ar.append(cp['resource']['intent'])
                    ar.append(cp['resource']['encounter']['reference'].strip().split('urn:uuid:')[1])
                    ar.append(cp['resource']['careTeam'][0]['reference'].strip().split('urn:uuid:')[1])

                    ar.append(activity['detail']['code']['coding'][0]['code'])
                    ar.append(activity['detail']['code']['coding'][0]['display'])
                    ar.append(activity['detail']['status'])
                    ar.append(activity['detail']['location']['display'])

                    arr.append(ar)
            else:
                ar = []
                ar.append(cp['resource']['id'])
                ar.append(cp['resource']['status'])
                ar.append(cp['resource']['subject']['reference'].strip().split('urn:uuid:')[1])
                ar.append(cp['resource']['period']['start'])

                if 'end' in cp['resource']['period'].keys():
                    ar.append(cp['resource']['period']['end'])
                else:
                    ar.append(np.nan)

                ar.append(cp['resource']['category'][0]['coding'][0]['code'])
                ar.append(cp['resource']['category'][1]['coding'][0]['code'])
                ar.append(cp['resource']['category'][1]['coding'][0]['display'])
                ar.append(cp['resource']['intent'])
                ar.append(cp['resource']['encounter']['reference'].strip().split('urn:uuid:')[1])
                ar.append(cp['resource']['careTeam'][0]['reference'].strip().split('urn:uuid:')[1])

                ar.append(np.nan)
                ar.append(np.nan)
                ar.append(np.nan)
                ar.append(np.nan)

                arr.append(ar)
    except Exception as e:
#         print(e)
        f_count += 1
        continue

end = time.time()
print(str(f_count)+' Files Failed...')
print(str(len(arr))+' Patient CarePlan bundles extracted as DataFrame in '+str(end-start)+ 'Seconds')

df_cp = pd.DataFrame(arr, columns = cols)

  0%|          | 0/2168 [00:00<?, ?it/s]

100%|██████████| 2168/2168 [03:42<00:00,  9.72it/s]

0 Files Failed...
17993 Patient CarePlan bundles extracted as DataFrame in 222.9869830608368Seconds


In [72]:
df_cp

,id,status,patientId,start,end,category,code,codeText,intent,encounter,careTeam,activityCode,activityCodeText,activityStatus,activityLocation
0,209c8400-573b-a7a7-239f-d0dd43c5fc99,active,209c8400-573b-a7a7-7791-657ca87065ba,1976-06-10T00:50:24-04:00,NaN,assess-plan,735985000,Diabetes self management plan (record artifact),order,209c8400-573b-a7a7-7add-d25aa886e071,50eeb6c4-91d4-af31-b46a-e5becd4eb910,160670007,Diabetic diet (finding),in-progress,BAKER COMMUNITY HEALTH CENTER INC
1,209c8400-573b-a7a7-239f-d0dd43c5fc99,active,209c8400-573b-a7a7-7791-657ca87065ba,1976-06-10T00:50:24-04:00,NaN,assess-plan,735985000,Diabetes self management plan (record artifact),order,209c8400-573b-a7a7-7add-d25aa886e071,50eeb6c4-91d4-af31-b46a-e5becd4eb910,229065009,Exercise therapy (regime/therapy),in-progress,BAKER COMMUNITY HEALTH CENTER INC
2,209c8400-573b-a7a7-0052-c30a2312e284,active,209c8400-573b-a7a7-7791-657ca87065ba,1988-04-14T00:50:24-04:00,NaN,assess-plan,736283006,Chronic obstructive pulmonary disease clinical...,order,209c8400-573b-a7a7-7c47-434bef6f2279,8f8c97c1-7779-4360-209c-e6bc935d7364,229065009,Exercise therapy (regime/therapy),in-progress,BAYSIDE DOCS URGENT CARE PLC
3,209c8400-573b-a7a7-0052-c30a2312e284,active,209c8400-573b-a7a7-7791-657ca87065ba,1988-04-14T00:50:24-04:00,NaN,assess-plan,736283006,Chronic obstructive pulmonary disease clinical...,order,209c8400-573b-a7a7-7c47-434bef6f2279,8f8c97c1-7779-4360-209c-e6bc935d7364,15081005,Pulmonary rehabilitation (regime/therapy),in-progress,BAYSIDE DOCS URGENT CARE PLC
4,209c8400-573b-a7a7-b1f9-0549988b5ebf,active,209c8400-573b-a7a7-7791-657ca87065ba,1991-08-01T00:50:24-04:00,NaN,assess-plan,736285004,Hyperlipidemia clinical management plan (recor...,order,209c8400-573b-a7a7-498b-205202abc924,4508d365-49c7-1e51-03cf-cca1bb7cc265,183063000,Low salt diet education (procedure),in-progress,DLP MARQUETTE GENERAL HOSPITAL LLC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17988,94e93cc8-fb91-b3fa-6f90-6f10795074b5,completed,94e93cc8-fb91-b3fa-f102-bd49abb14d72,2023-04-28T23:24:19-04:00,2023-11-10T22:24:19-05:00,assess-plan,134435003,Routine antenatal care (regime/therapy),order,94e93cc8-fb91-b3fa-8ae9-607e761c7a2c,05159a40-d5f1-9abe-5fdb-4b7089cfbe3b,135892000,Antenatal education (procedure),completed,PORTAGE HOSPITAL LLC
17989,94e93cc8-fb91-b3fa-6f90-6f10795074b5,completed,94e93cc8-fb91-b3fa-f102-bd49abb14d72,2023-04-28T23:24:19-04:00,2023-11-10T22:24:19-05:00,assess-plan,134435003,Routine antenatal care (regime/therapy),order,94e93cc8-fb91-b3fa-8ae9-607e761c7a2c,05159a40-d5f1-9abe-5fdb-4b7089cfbe3b,713076009,Antenatal risk assessment (procedure),completed,PORTAGE HOSPITAL LLC
17990,94e93cc8-fb91-b3fa-6f90-6f10795074b5,completed,94e93cc8-fb91-b3fa-f102-bd49abb14d72,2023-04-28T23:24:19-04:00,2023-11-10T22:24:19-05:00,assess-plan,134435003,Routine antenatal care (regime/therapy),order,94e93cc8-fb91-b3fa-8ae9-607e761c7a2c,05159a40-d5f1-9abe-5fdb-4b7089cfbe3b,312404004,Antenatal blood tests (procedure),completed,PORTAGE HOSPITAL LLC
17991,94e93cc8-fb91-b3fa-a53f-fba731eebe4d,active,94e93cc8-fb91-b3fa-f102-bd49abb14d72,2025-05-09T23:24:19-04:00,NaN,assess-plan,736285004,Hyperlipidemia clinical management plan (recor...,order,94e93cc8-fb91-b3fa-2205-0ab84dd6ce9a,131d33e7-52c3-af69-a803-f294b59c4920,183063000,Low salt diet education (procedure),in-progress,PORTAGE HOSPITAL LLC


In [73]:
df_cp.to_csv(output_folder_path+delim+'CarePlan.csv')
del df_cp
gc.collect()

0